# Semantic Search Deep-Dive

This notebook provides an in-depth exploration of the semantic search
capabilities of `indianconstitution`, powered by
[sentence-transformers](https://www.sbert.net/).

**Topics covered:**
1. Basic semantic search
2. Keyword vs. semantic comparison
3. Multi-query batch analysis
4. Similarity score interpretation
5. Building semantic clusters

**Requirements:**
```bash
pip install "indianconstitution[ai,data]"
```

In [ ]:
# Uncomment to install:
# !pip install -q "indianconstitution[ai,data]"

In [ ]:
from indianconstitution import get_constitution

ic = get_constitution()
print(f"Loaded: {ic}")

## 1. Basic Semantic Search

The semantic engine uses `all-MiniLM-L6-v2` to encode both the query and
all article texts. Cosine similarity is used for ranking.

In [ ]:
results = ic.semantic_search(
    "protection against arbitrary state action",
    top_k=5
)

for r in results:
    print(f"[Art. {r.number}] {r.title}")
    print(f"  Score: {r.score:.4f}")
    print(f"  Preview: {r.content[:120]}...\n")

## 2. Keyword vs. Semantic Comparison

Semantic search finds conceptually relevant results even when
exact keywords don't match.

In [ ]:
test_queries = [
    "can the police arrest someone without a warrant?",
    "rights of minorities in education",
    "when can the president declare emergency?",
    "how are judges appointed?",
]

for query in test_queries:
    print(f"\n{'\u2550' * 60}")
    print(f"  Query: {query}")
    print(f"{'\u2550' * 60}")
    
    kw = ic.search(query, limit=3)
    sem = ic.semantic_search(query, top_k=3)
    
    print("  Keyword results:")
    if kw:
        for r in kw:
            print(f"    [{r.number}] {r.title}")
    else:
        print("    (no matches)")
    
    print("  Semantic results:")
    for r in sem:
        print(f"    [{r.number}] {r.title}  (score: {r.score:.4f})")

## 3. Batch Query Analysis

Analyse multiple legal concepts and see which articles are most relevant.

In [ ]:
import pandas as pd

concepts = [
    "freedom of speech",
    "right to equality",
    "right to life",
    "religious freedom",
    "property rights",
    "labour protections",
    "judicial independence",
    "federal structure",
]

rows = []
for concept in concepts:
    top = ic.semantic_search(concept, top_k=1)[0]
    rows.append({
        "Concept": concept,
        "Top Article": f"Art. {top.number}",
        "Title": top.title,
        "Score": f"{top.score:.4f}",
    })

pd.DataFrame(rows)

## 4. Score Distribution Analysis

Understanding what the similarity scores mean.

In [ ]:
# Get scores for all articles for a query
results = ic.semantic_search("right to education", top_k=len(ic))

scores = [r.score for r in results]
print(f"Score range: [{min(scores):.4f}, {max(scores):.4f}]")
print(f"Mean score:  {sum(scores)/len(scores):.4f}")
print(f"Median score: {sorted(scores)[len(scores)//2]:.4f}")

# Threshold analysis
for threshold in [0.5, 0.4, 0.3, 0.2]:
    above = sum(1 for s in scores if s >= threshold)
    print(f"  Articles with score >= {threshold}: {above}")

## 5. Semantic Clusters

Group articles by semantic similarity to key constitutional themes.

In [ ]:
themes = {
    "Civil Liberties": "individual freedom speech religion assembly",
    "Governance": "parliament president executive administration",
    "Judiciary": "supreme court high court judges justice",
    "Federalism": "states union territories division of powers",
    "Social Justice": "equality discrimination caste backward classes",
}

for theme_name, theme_query in themes.items():
    print(f"\n--- {theme_name} ---")
    results = ic.semantic_search(theme_query, top_k=5)
    for r in results:
        print(f"  [{r.number}] {r.title}  ({r.score:.3f})")